# Robô de triagem da Cobratec — modelo rodando no Colab

Sobe um **Ollama com GPU** aqui no Colab e o entrega ao inventário por um túnel
autenticado. Serve para a máquina do escritório não ter que segurar o modelo.

```
WhatsApp → WAHA → inventário (/chat) → túnel → [Colab] proxy → Ollama (GPU)
```

## Leia antes de rodar

**Isto é o caminho de TESTE.** A decisão 31 escolheu rodar o modelo dentro da
empresa porque a fala do devedor não deve sair dela. Apontar o inventário para
cá manda a mensagem do devedor para uma VM do Google — o que é aceitável para
**medir modelo e afinar a triagem com mensagens inventadas**, e não é aceitável
com devedor real. A tela `/chat → Conexão` avisa em vermelho quando o modelo
está fora da rede; se o aviso estiver aparecendo em produção, algo está errado.

O que o robô faz continua o mesmo, aqui ou lá: ele **não fala de valor, acordo
nem pagamento** — isso é barrado por código no inventário (`lib/chat-bot.ts`),
não pelo modelo. Trocar de máquina não afrouxa nada disso.

**Limites do Colab, que você vai encontrar:** a sessão cai sozinha depois de
~90 min sem uso e tem teto de ~12h; o endereço do túnel **muda a cada vez** que
você roda de novo (e o `.env` do inventário precisa ser atualizado); e servir
tráfego contínuo não é o uso que o Colab se propõe a suportar. Para valer,
o modelo mora numa máquina sua.

## Como usar

1. **Ambiente de execução → Alterar o tipo → GPU (T4)**. Sem GPU não vale a
   pena: a CPU do Colab é mais lenta que a do escritório.
2. Rode as células **1 a 4** em ordem.
3. Copie o bloco que a célula 4 imprime para o `.env` do inventário e recrie o
   app (`docker compose up -d`).
4. Deixe a célula 5 rodando e **a aba aberta** — é o que segura a sessão viva.

## 1. GPU e instalação do Ollama

In [ ]:
import shutil, subprocess

# Sem GPU o Colab não ajuda em nada: a CPU dele é mais fraca que a de um
# desktop de escritório. Melhor descobrir agora que depois de baixar 2GB.
#
# `shutil.which` e não `subprocess.run` direto: numa sessão sem GPU o
# nvidia-smi NÃO EXISTE, e chamá-lo levanta FileNotFoundError em vez de
# devolver código de erro. Procurar o binário antes evita transformar
# "faltou escolher a GPU" num traceback que parece defeito do notebook.
if not shutil.which("nvidia-smi"):
    raise SystemExit(
        "SEM GPU nesta sessão.\n"
        "Ambiente de execução → Alterar o tipo de ambiente → T4 GPU → Salvar,\n"
        "e rode esta célula de novo. Em CPU o Colab é mais lento que a máquina\n"
        "do escritório, então não vale a pena continuar."
    )

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
)
if gpu.returncode != 0:
    raise SystemExit(f"nvidia-smi respondeu erro: {gpu.stderr.strip()}")
print("GPU:", gpu.stdout.strip())


def instalado():
    return shutil.which("ollama") is not None


# O pacote do Ollama vem comprimido em zstd, e a imagem do Colab NÃO traz o
# descompressor. Sem ele o instalador oficial para e imprime "instale o zstd" —
# que foi exatamente onde esta célula morreu antes. Vem primeiro porque os dois
# caminhos de instalação dependem dele.
if not shutil.which("unzstd"):
    print("Instalando o zstd (o Colab não traz, e o pacote do Ollama precisa)…")
    if subprocess.run(["apt-get", "-qq", "install", "-y", "zstd"]).returncode != 0:
        subprocess.run(["apt-get", "-qq", "update"])
        subprocess.run(["apt-get", "-qq", "install", "-y", "zstd"])
    if not shutil.which("unzstd"):
        raise SystemExit("não consegui instalar o zstd — rode a célula de novo")

# Instalação por dois caminhos, e conferindo o resultado em vez do código de
# saída. Isto NÃO é excesso de zelo: o script oficial baixa ~1,4 GB de um CDN
# que já resetou a conexão no meio (`curl: (56)`) aqui no Colab. Pior, a falha é
# silenciosa quando se usa `!curl ... | sh`: o magic falha, o Python segue, e o
# erro só aparece na célula seguinte como "ollama não encontrado".
if not instalado():
    print("Instalando pelo script oficial…")
    subprocess.run(
        "curl -fsSL --retry 3 --retry-all-errors https://ollama.com/install.sh | sh",
        shell=True,
    )

if not instalado():
    # Mesmo pacote, outro caminho: o release do GitHub costuma passar quando o
    # CDN do ollama.com cai. `--retry-all-errors -C -` retoma de onde parou em
    # vez de recomeçar 1,4 GB.
    print("O script oficial não concluiu. Baixando o pacote do GitHub…")
    subprocess.run(
        ["curl", "-fL", "--retry", "5", "--retry-delay", "3", "--retry-all-errors",
         "-C", "-", "-o", "/tmp/ollama.tar.zst",
         "https://github.com/ollama/ollama/releases/latest/download/"
         "ollama-linux-amd64.tar.zst"],
        check=True,
    )
    # O pacote traz `bin/ollama` na raiz: extraído aqui, cai em /usr/local/bin,
    # que já está no PATH.
    subprocess.run(
        ["tar", "--use-compress-program=unzstd", "-xf", "/tmp/ollama.tar.zst",
         "-C", "/usr/local"],
        check=True,
    )

if not instalado():
    raise SystemExit(
        "não consegui instalar o Ollama pelos dois caminhos.\n"
        "Quase sempre é rede do Colab: espere um minuto e rode a célula de novo."
    )
print("Ollama instalado em", shutil.which("ollama"))

## 2. Subir o Ollama e baixar o modelo

`llama3.2:3b` é o padrão aqui **porque tem GPU**. Na máquina do escritório o
padrão é o `1b`, que cabe em CPU. É justamente essa diferença que este notebook
existe para você medir: se o 3B com GPU não triar visivelmente melhor que o 1B
local, não vale a dependência de um túnel.

In [ ]:
import os, shutil, time, subprocess, requests

MODELO = "llama3.2:3b"  # troque para medir: llama3.2:1b, qwen2.5:3b, gemma2:2b
OLLAMA = "http://127.0.0.1:11434"

# Mesma armadilha da célula anterior: binário ausente levanta exceção, não
# código de erro. Se a instalação falhou, o erro tem que dizer isso.
if not shutil.which("ollama"):
    raise SystemExit("O Ollama não está instalado — rode a célula 1 primeiro.")

os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
# Mantém o modelo na GPU entre mensagens. Sem isto, cada pausa na conversa paga
# a carga de novo — o mesmo motivo do keep_alive em lib/chat-bot.ts.
os.environ["OLLAMA_KEEP_ALIVE"] = "60m"

servidor = subprocess.Popen(["ollama", "serve"],
                            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(60):
    if servidor.poll() is not None:
        raise SystemExit("o `ollama serve` morreu ao subir — rode a célula 1 de novo")
    try:
        requests.get(f"{OLLAMA}/api/tags", timeout=1)
        break
    except Exception:
        time.sleep(1)
else:
    raise SystemExit("o Ollama não respondeu em 60s")

print("Ollama de pé. Baixando", MODELO, "— alguns minutos na primeira vez.")
puxar = subprocess.run(["ollama", "pull", MODELO], capture_output=True, text=True)

# Conferir a LISTA, não o código de saída: é o que prova que o modelo está
# mesmo disponível para a próxima célula. Nome de modelo errado é o engano mais
# fácil aqui, e ele falha tarde — só quando a medição não encontra o modelo.
baixados = [m["name"] for m in requests.get(f"{OLLAMA}/api/tags").json()["models"]]
if not any(m == MODELO or m.startswith(f"{MODELO}:") for m in baixados):
    raise SystemExit(
        f"o modelo {MODELO} não ficou disponível.\n"
        f"{(puxar.stderr or puxar.stdout).strip()[-300:]}\n"
        f"baixados agora: {baixados or 'nenhum'}"
    )
print("pronto:", MODELO)

## 3. Medir antes de confiar

Roda o modelo com as falas que mais aparecem e mostra **quanto tempo cada uma
leva**. O inventário desiste em **45s** e manda para a fila, então qualquer
resposta acima disso é uma conversa que a operadora vai atender de qualquer
jeito — só que depois de a pessoa esperar.

Repare que as falas graves ("quanto eu devo") **nem chegariam ao modelo** em
produção: o inventário as manda para a fila antes, sem inferência. Estão aqui
só para você ver o que o modelo faria se dependesse dele — e é por isso que não
depende.

In [ ]:
import json, time, requests

FALAS = [
    "oi",
    "bom dia, tudo bem?",
    "quem fala?",
    "o que é a cobratec?",
    "quanto eu devo?",             # em produção: fila, sem passar pelo modelo
    "já paguei isso mês passado",  # em produção: fila, sem passar pelo modelo
]

# Cópia enxuta do prompt de lib/chat-bot.ts, só para a medição ser honesta:
# prompt curto é parte do desenho, e medir com outro texto mediria outra coisa.
SISTEMA = (
    "Você é a recepcionista virtual da Cobratec no WhatsApp. Fale com a pessoa, "
    "com educação.\n\nVocê não tem acesso a dado nenhum: nem cadastro, nem valor, "
    "nem prazo. Nunca invente.\n\nEscale para uma atendente humana sempre que o "
    "assunto for a dívida ou a pessoa estiver irritada.\n\nResponda SEMPRE em JSON: "
    '{"resposta":"o que dizer","escalar":true ou false,"motivo":"por que escalou"}'
)

print(f"{'fala':<32} {'seg':>6}  resposta")
print("-" * 92)
piores = []
for fala in FALAS:
    t0 = time.time()
    r = requests.post(f"{OLLAMA}/api/chat", timeout=180, json={
        "model": MODELO,
        "messages": [{"role": "system", "content": SISTEMA},
                     {"role": "user", "content": fala}],
        "stream": False, "format": "json", "keep_alive": "60m",
        "options": {"temperature": 0, "num_predict": 120},
    })
    seg = time.time() - t0
    piores.append(seg)
    try:
        d = json.loads(r.json()["message"]["content"])
        saida = f"{'ESCALA' if d.get('escalar') else 'responde'}: {d.get('resposta', '')[:44]}"
    except Exception:
        saida = f"FORA DO FORMATO (o inventário escalaria): {r.text[:40]}"
    print(f"{fala:<32} {seg:>6.1f}  {saida}")

pior = max(piores)
print("-" * 92)
print(f"pior caso: {pior:.1f}s  (teto do inventário: 45s)")
print("OK: cabe com folga." if pior < 20 else
      ("Apertado: passe para um modelo menor." if pior < 45 else
       "NÃO SERVE: acima do teto, tudo cairia na fila."))

## 4. Abrir a porta para o inventário

O Ollama **não tem autenticação nenhuma**. Num túnel público isso seria um
modelo aberto para quem achasse o endereço, e o endereço não é secreto.

Então quem atende o túnel não é o Ollama: é um proxy de vinte linhas que exige
um `Bearer` e só deixa passar **duas rotas** — conversar e listar modelos. Sem
isso, um `DELETE /api/delete` da internet apagaria o modelo no meio do
atendimento.

In [ ]:
import hmac, queue, re, secrets, shutil, socket, subprocess, sys, threading, time, requests

try:
    from flask import Flask, request, Response
except ImportError:
    # Python puro em vez de `!pip`: magic indentado dentro de `except` depende
    # do IPython transformar a linha, e isto aqui precisa funcionar sempre.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "flask"], check=True)
    from flask import Flask, request, Response

TOKEN = secrets.token_urlsafe(32)

# Só o que o inventário usa. Lista de permissão, não de bloqueio: rota nova do
# Ollama nasce fechada em vez de nascer exposta.
LIBERADAS = {("POST", "/api/chat"), ("GET", "/api/tags")}

app = Flask(__name__)

@app.route("/<path:caminho>", methods=["GET", "POST"])
def repassar(caminho):
    rota = "/" + caminho
    if (request.method, rota) not in LIBERADAS:
        return Response('{"error":"rota fechada"}', 403, mimetype="application/json")

    # compare_digest: comparação de segredo não vaza o tamanho do acerto.
    enviado = request.headers.get("Authorization", "")
    if not hmac.compare_digest(enviado, f"Bearer {TOKEN}"):
        return Response('{"error":"token inválido"}', 401, mimetype="application/json")

    resp = requests.request(request.method, OLLAMA + rota,
                            data=request.get_data(), timeout=180,
                            headers={"content-type": "application/json"})
    return Response(resp.content, resp.status_code, mimetype="application/json")

# Porta escolhida na hora, e não fixa em 8000: reexecutar esta célula com o
# Flask anterior ainda de pé daria "address already in use", o servidor novo não
# subiria e o túnel apontaria para o antigo — que tem OUTRO token. O sintoma
# seria 401 no inventário, sem nada errado à vista.
with socket.socket() as s:
    s.bind(("127.0.0.1", 0))
    PORTA = s.getsockname()[1]

threading.Thread(
    target=lambda: app.run(host="127.0.0.1", port=PORTA, threaded=True),
    daemon=True,
).start()

# Espera o servidor atender de fato. `time.sleep` fixo é aposta: em máquina
# carregada o túnel subiria antes do proxy existir.
for _ in range(50):
    try:
        requests.get(f"http://127.0.0.1:{PORTA}/api/tags", timeout=1)
        break
    except Exception:
        time.sleep(0.2)
else:
    raise SystemExit("o proxy não subiu — rode a célula de novo")

# Túnel do Cloudflare: endereço público sem conta e sem cadastro. Ele MUDA a
# cada execução — é a fricção principal deste caminho.
if not shutil.which("cloudflared"):
    subprocess.run(
        ["curl", "-fL", "--retry", "5", "--retry-delay", "2", "--retry-all-errors",
         "-o", "/tmp/cf.deb",
         "https://github.com/cloudflare/cloudflared/releases/latest/download/"
         "cloudflared-linux-amd64.deb"],
        check=True,
    )
    subprocess.run(["dpkg", "-i", "/tmp/cf.deb"], capture_output=True)

if not shutil.which("cloudflared"):
    raise SystemExit("o cloudflared não instalou — rode a célula de novo")

tunel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORTA}", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

# Ler o stdout do túnel direto no `for` penduraria a célula PARA SEMPRE se o
# cloudflared subisse mudo (rede bloqueada, serviço fora do ar). Uma thread
# despeja as linhas numa fila e aqui se espera com prazo: o notebook precisa
# poder desistir e dizer o motivo.
linhas = queue.Queue()
threading.Thread(target=lambda: [linhas.put(l) for l in tunel.stdout],
                 daemon=True).start()

url, prazo = None, time.time() + 60
while time.time() < prazo:
    try:
        achou = re.search(r"https://[-\w]+\.trycloudflare\.com",
                          linhas.get(timeout=5))
    except queue.Empty:
        if tunel.poll() is not None:
            raise SystemExit("o cloudflared morreu ao subir — rode a célula de novo")
        continue
    if achou:
        url = achou.group(0)
        break

if not url:
    tunel.terminate()
    raise SystemExit("o túnel não anunciou endereço em 60s — rode a célula de novo")

conferencia = requests.get(f"{url}/api/tags", timeout=30,
                           headers={"Authorization": f"Bearer {TOKEN}"})
print("túnel respondendo:", conferencia.status_code == 200)
print("porta fechada sem token:", requests.get(f"{url}/api/tags", timeout=30).status_code == 401)

print("\n" + "=" * 74)
print("Cole no .env do inventário e rode:  docker compose up -d")
print("=" * 74)
print(f'OLLAMA_URL="{url}"')
print(f'OLLAMA_MODELO="{MODELO}"')
print(f'OLLAMA_TOKEN="{TOKEN}"')
print("=" * 74)
print("Este endereço morre quando a sessão do Colab cair. Quando isso acontecer,")
print("rode o notebook de novo e troque as três linhas — ou apague OLLAMA_URL,")
print("que o atendimento volta a cair na fila da operadora sem quebrar nada.")

## 5. Segurar a sessão viva

Deixe esta célula rodando **e a aba aberta**. Ela também é o seu monitor: mostra
quantas mensagens o inventário mandou e avisa quando o modelo cair.

Para desligar tudo: interrompa a célula e feche a aba. No inventário, apague
`OLLAMA_URL` do `.env` — o atendimento volta inteiro para a fila da operadora,
sem quebrar nada.

In [ ]:
import time, requests
from datetime import datetime, timedelta, timezone

BRASIL = timezone(timedelta(hours=-3))
inicio = time.time()

while True:
    try:
        vivo = requests.get(f"{OLLAMA}/api/tags", timeout=5).status_code == 200
    except Exception:
        vivo = False

    horas = (time.time() - inicio) / 3600
    agora = datetime.now(BRASIL).strftime("%H:%M")
    estado = "ok" if vivo else "MODELO FORA DO AR — o inventário está escalando tudo"
    print(f"\r{agora}  de pé há {horas:4.1f}h  {estado}   ", end="")

    if horas > 11.5:
        print("\nperto do teto de 12h do Colab: a sessão vai cair em breve.")
    time.sleep(60)